# Notebook 04: Enterotypes vs Transcriptome

Test whether enterotypes associate with gene expression.

**Seven comparisons:**

- **Part 8:** Enterotype vs gResid (3 locations/tissues and 6 immune cell types)
- **Part 9:** Enterotype vs module eigengenes + genes (3 locations/tissues and 6 immune cell types)

All sections use `corr.method = 'kruskal'` (Kruskal-Wallis for categorical enterotypes).

## Bootstrap: Environment Setup

In [1]:
# --- Bootstrap: replicate 00_setup.ipynb environment ---
BASE_DIR <- normalizePath(file.path(getwd(), '..'))
DATA_DIR    <- file.path(BASE_DIR, 'Data')
CODE_DIR    <- file.path(BASE_DIR, 'code', 'CEDAR-master')
OUTPUT_DIR  <- file.path(BASE_DIR, 'notebooks', 'output')
PLOTS_DATA  <- file.path(BASE_DIR, 'code', 'data_for_plots')
TRANSCR_DIR       <- file.path(DATA_DIR, 'CEDAR1_transcriptome')
EXPR_PC_CORR_DIR  <- file.path(TRANSCR_DIR, 'Expr_data_Corr_for_PCs')
EXPR_NO_PC_DIR    <- file.path(TRANSCR_DIR, 'Expr_data_Not_corr_for_PCs')
TRANSCR_PCS_DIR   <- file.path(TRANSCR_DIR, 'Momo-Dmitrieva2018')
MODULE_EIGEN_DIR  <- file.path(TRANSCR_DIR, 'Module_eigengenes_expr')
dir.create(OUTPUT_DIR, showWarnings = FALSE, recursive = TRUE)

suppressPackageStartupMessages({
    library(data.table)
    library(ade4)
    library(vegan)
    library(chemometrics)
    library(ggplot2)
    library(igraph)
    library(gplots)
    library(ape)
    library(stringr)
    library(jsonlite)
    library(dplyr)
    library(lattice)
    library(RhpcBLASctl)
    library(data.table)
    library(parallel)
    library(doParallel)
    library(foreach)
})
for (pkg in c('WGCNA', 'Rtsne', 'DirichletMultinomial', 'WebGestaltR', 'RCurl')) {
    if (requireNamespace(pkg, quietly = TRUE)) {
        suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
}

R_DIR <- file.path(BASE_DIR, 'notebooks', 'R')
source(file.path(R_DIR, 'functions_general.R'))
source(file.path(R_DIR, 'functions_figures.R'))
source(file.path(R_DIR, 'functions_association.R'))
source(file.path(R_DIR, 'functions_transcriptome.R'))

LOCATIONS   <- c('IL', 'TR', 'RE')
CELL_TYPES  <- c('CD4', 'CD8', 'CD14', 'CD15', 'CD19', 'PLA')
ALL_TYPES   <- c(LOCATIONS, CELL_TYPES)
N_TRANSCR_PCS_LOCATION <- c(IL = 59, TR = 50, RE = 53)
N_TRANSCR_PCS_CELL_TYPE <- c(CD4 = 38, CD8 = 35, CD14 = 36,
                          CD15 = 27, CD19 = 40, PLA = 23)
N_TRANSCR_PCS <- c(N_TRANSCR_PCS_LOCATION, N_TRANSCR_PCS_CELL_TYPE)

N_SIDAK_PERM = 1000

#cat('Bootstrap complete. BASE_DIR:', BASE_DIR, '\n')

Loading required package: fitdistrplus

Loading required package: MASS


Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select


Loading required package: survival

Loading required package: tibble


Attaching package: ‘tibble’


The following object is masked from ‘package:igraph’:

    as_data_frame




In [2]:
# --- Setup Parallel Backend ---
num_cores <- 30

## Load Input Data

In [3]:
# 3 min
transcriptome_list_residuals <- list()
for(tn in ALL_TYPES){
    fname <- paste0(word(tn, 1, sep = '\\.'), '_GE_Corrected4_Covars_PCs.txt')
    transcriptome_list_residuals[[tn]] <- read.table(file.path(EXPR_PC_CORR_DIR, fname), row.names = 1, header = T)
}

In [4]:
# Output subdirectory for this notebook
NB04_OUT <- file.path(OUTPUT_DIR, 'nb04')
dir.create(NB04_OUT, showWarnings = FALSE, recursive = TRUE)

In [5]:
# --- Load enterotype assignments ---
enterotypes.df <- read.table.smart(file.path(DATA_DIR, 'Samples_averagedLocAmpl.tsv'))
enterotypes.df[, 1] <- as.factor(enterotypes.df[, 1])
colnames(enterotypes.df) <- 'ent'
cat('Enterotypes:', nrow(enterotypes.df), 'samples\n')
cat('Enterotype distribution:\n')
print(table(enterotypes.df$ent))

# --- Load probe-to-gene mapping ---
ProbeIDs_to_genes.table <- read.table.smart(
    file.path(DATA_DIR, 'Expression_array_ProbeIDs_to_genes.tsv'))
cat('\nProbe-gene mapping:', nrow(ProbeIDs_to_genes.table), 'probes\n')

Enterotypes: 307 samples
Enterotype distribution:

 1  2  3  4 
89 82 66 66 

Probe-gene mapping: 31137 probes


## Part 8a: Enterotype vs Gene Expression Residuals (3 locations)

Kruskal-Wallis test of enterotype against each individual gene
expression residual (PC-corrected).

- Expression residuals from `Expr_data_Corr_for_PCs/`
- `number.of.PCs.transcriptome = NULL` → all columns used (minus ID column)
- `transcr.exclude.mode = 'first_1_column'` → exclude ID column only
- `remove.outliers = c(F, T)`

In [6]:
# Prepare
prep_8 <- prepare_data_universal_ent(
    microb_df = enterotypes.df,
    transcr_list = transcriptome_list_residuals[c(LOCATIONS, CELL_TYPES)],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE)
)

In [8]:
# Tissues (Part 8a)
pval_8a <- run_universal_association_ent(
    prep_8[LOCATIONS],
    inflation_correct=TRUE
)

In [9]:
res_8a <- run_neff_pipeline_ent(
    pval_observed = pval_8a,
    prepared_data = prep_8[LOCATIONS],
    microb_base = enterotypes.df, 
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda=attr(pval_8a, "lambda")    
)

Starting Neff estimation (Kruskal-Wallis) with 1000 permutations...
Neff coefficient of variation:  0.0085 
Total Tests: 45327 
Effective Independent Tests (Neff): 25515.11 
Reduction in penalty: 43.7 %

p Sidak <= 0.15: 1 
RE.Ent.X3840398 
      0.1406603 

p Hochberg <= 0.15: 0 
named numeric(0)


In [10]:
write.pvalue.vectors.to.file(
    file.path(NB04_OUT, "pvals_Part8a_Ent_vs_gResid_3locs.tsv"),
    Original = pval_8a,
    Hochberg = res_8a$pval_hochberg,
    Sidak = res_8a$pval_sidak, 
    signif_=4
)

## Part 8b: Enterotype vs Gene Expression Residuals (6 immune cell types)

In [11]:
pval_8b <- run_universal_association_ent(
    prep_8[CELL_TYPES],
    inflation_correct=TRUE
)

In [12]:
res_8b <- run_neff_pipeline_ent(
    pval_observed = pval_8b,
    prepared_data = prep_8[CELL_TYPES],
    microb_base = enterotypes.df, 
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda=attr(pval_8b, "lambda")    
)

Starting Neff estimation (Kruskal-Wallis) with 1000 permutations...
Neff coefficient of variation:  0.0269 
Total Tests: 70308 
Effective Independent Tests (Neff): 55863.99 
Reduction in penalty: 20.5 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 0 
named numeric(0)


In [13]:
write.pvalue.vectors.to.file(
    file.path(NB04_OUT, "pvals_Part8b_Ent_vs_gResid_6cells.tsv"),
    Original = pval_8b,
    Hochberg = res_8b$pval_hochberg,
    Sidak = res_8b$pval_sidak, 
    signif_=4
)

## Part 9a: Enterotype vs Module Eigengenes + Genes (3 Locations)

Kruskal-Wallis test of enterotype against each column in the
module eigengene files (which contain BOTH individual probes AND ME columns).

- Module eigengene files from `Module_eigengenes_expr/`
- `remove.outliers = c(F, F)` — NO outlier removal on either side
- `remove.outliers.IQR = 5`
- No inflation correction
- `transcr.exclude.mode = 'first_1_column'` — exclude only the ID column

In [14]:
# Prepare
prep_9 <- prepare_data_universal_ent(
    microb_df = enterotypes.df,
    transcr_list = transcriptome_list_residuals[c(LOCATIONS, CELL_TYPES)],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE)
)

In [15]:
# TODO check I am ok with IQR 1.5 not 5 !

In [16]:
# Tissues (Part 9a)
pval_9a <- run_universal_association_ent(
    prep_9[LOCATIONS],
    inflation_correct=TRUE
)

In [17]:
res_9a <- run_neff_pipeline_ent(
    pval_observed = pval_9a,
    prepared_data = prep_9[LOCATIONS],
    microb_base = enterotypes.df, 
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda=attr(pval_9a, "lambda")    
)

Starting Neff estimation (Kruskal-Wallis) with 1000 permutations...
Neff coefficient of variation:  0.0232 
Total Tests: 45327 
Effective Independent Tests (Neff): 23016.96 
Reduction in penalty: 49.2 %

p Sidak <= 0.15: 1 
RE.Ent.X3840398 
      0.1278108 

p Hochberg <= 0.15: 0 
named numeric(0)


In [18]:
write.pvalue.vectors.to.file(
    file.path(NB04_OUT, "pvals_Part9a_Ent_vs_gResid_3locs.tsv"),
    Original = pval_9a,
    Hochberg = res_9a$pval_hochberg,
    Sidak = res_9a$pval_sidak, 
    signif_=4
)

## Part 9b: Enterotype vs Module Eigengenes + Genes (6 immune cell types)

In [19]:
pval_9b <- run_universal_association_ent(
    prep_9[CELL_TYPES],
    inflation_correct=TRUE
)

In [20]:
res_9b <- run_neff_pipeline_ent(
    pval_observed = pval_9b,
    prepared_data = prep_9[CELL_TYPES],
    microb_base = enterotypes.df, 
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda=attr(pval_9b, "lambda")    
)

Starting Neff estimation (Kruskal-Wallis) with 1000 permutations...
Neff coefficient of variation:  0.0266 
Total Tests: 70308 
Effective Independent Tests (Neff): 56732.2 
Reduction in penalty: 19.3 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 0 
named numeric(0)


In [21]:
write.pvalue.vectors.to.file(
    file.path(NB04_OUT, "pvals_Part9b_Ent_vs_gResid_6cells.tsv"),
    Original = pval_9b,
    Hochberg = res_9b$pval_hochberg,
    Sidak = res_9b$pval_sidak, 
    signif_=4
)